In [2]:
import pandas as pd

file_path = "../0.data/criteo-uplift-v2.1.csv"

df = pd.read_csv(file_path)

print(df.shape)
print(df.columns.tolist())
df.head()

(13979592, 16)
['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'treatment', 'conversion', 'visit', 'exposure']


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [3]:
print(df.columns.tolist())

['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'treatment', 'conversion', 'visit', 'exposure']


In [4]:
df.head()

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [5]:
print(df.dtypes)

print(df[["treatment", "conversion", "visit", "exposure"]].value_counts())

print(df[["treatment", "conversion", "visit", "exposure"]].mean())

print(df.isna().sum())

f0            float64
f1            float64
f2            float64
f3            float64
f4            float64
f5            float64
f6            float64
f7            float64
f8            float64
f9            float64
f10           float64
f11           float64
treatment       int64
conversion      int64
visit           int64
exposure        int64
dtype: object
treatment  conversion  visit  exposure
1          0           0      0           11055129
0          0           0      0            2016832
1          0           1      0             385634
                       0      1             250702
                       1      1             154479
0          0           1      0              76042
1          1           1      1              23031
                              0              13680
0          1           1      0               4063
Name: count, dtype: int64
treatment     0.850000
conversion    0.002917
visit         0.046992
exposure      0.030631
dtype: float64
f0 

In [7]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from econml.dml import CausalForestDML

In [8]:
feature_cols = [f"f{i}" for i in range(12)]

sample_df = df.sample(
    n=500_000,
    random_state=42
).copy()

# 処置×アウトカムの構成比を保つ
sample_df["strata"] = (
    sample_df["treatment"].astype(str)
    + "_"
    + sample_df["conversion"].astype(str)
)

train_df = df.sample(
    n=1_000_000,
    random_state=42
).copy()

test_df = df.drop(index=train_df.index).sample(
    n=3_000_000,
    random_state=43
).copy()

X_train = train_df[feature_cols].to_numpy()
W_train = train_df["treatment"].to_numpy()
Y_train = train_df["conversion"].to_numpy()

X_test = test_df[feature_cols].to_numpy()
W_test = test_df["treatment"].to_numpy()
Y_test = test_df["conversion"].to_numpy()

In [9]:
cf = CausalForestDML(
    discrete_treatment=True,
    discrete_outcome=True,

    n_estimators=400,
    min_samples_leaf=100,
    max_depth=None,
    max_samples=0.45,

    honest=True,
    inference=True,

    cv=3,
    n_jobs=-1,
    random_state=42
)

cf.fit(
    Y_train,
    W_train,
    X=X_train
)

/opt/anaconda3/envs/thesis311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/envs/thesis311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.ht

In [10]:
cate_hat = cf.effect(X_test)

test_result = test_df[
    feature_cols + ["treatment", "conversion", "visit", "exposure"]
].copy()

test_result["cate_hat"] = cate_hat

test_result["cate_hat"].describe()

count    3.000000e+06
mean     1.005989e-03
std      4.749176e-03
min     -2.599943e-02
25%     -7.379819e-07
50%      7.131042e-06
75%      5.242193e-04
max      8.742833e-02
Name: cate_hat, dtype: float64

In [11]:
test_result["cate_group"] = pd.qcut(
    test_result["cate_hat"],
    q=5,
    labels=["Q1_low", "Q2", "Q3", "Q4", "Q5_high"],
    duplicates="drop"
)

group_effect = (
    test_result
    .groupby(["cate_group", "treatment"], observed=True)["conversion"]
    .agg(["mean", "count"])
    .reset_index()
)

group_means = group_effect.pivot(
    index="cate_group",
    columns="treatment",
    values="mean"
)

group_counts = group_effect.pivot(
    index="cate_group",
    columns="treatment",
    values="count"
)

group_means["observed_itt"] = (
    group_means[1] - group_means[0]
)

print(group_means)
print(group_counts)

treatment          0         1  observed_itt
cate_group                                  
Q1_low      0.000649  0.000807      0.000158
Q2          0.000044  0.000061      0.000017
Q3          0.000099  0.000159      0.000060
Q4          0.000297  0.000595      0.000299
Q5_high     0.008956  0.013380      0.004424
treatment       0       1
cate_group               
Q1_low      90843  509166
Q2          90328  509667
Q3          90748  509248
Q4          91022  508978
Q5_high     86865  513135


In [12]:
group_summary = (
    test_result
    .groupby(["cate_group", "treatment"], observed=True)["conversion"]
    .agg(
        conversion_rate="mean",
        conversions="sum",
        n="count"
    )
    .reset_index()
)

group_summary

,cate_group,treatment,conversion_rate,conversions,n
0,Q1_low,0,0.000649,59,90843
1,Q1_low,1,0.000807,411,509166
2,Q2,0,0.000044,4,90328
3,Q2,1,0.000061,31,509667
4,Q3,0,0.000099,9,90748
5,Q3,1,0.000159,81,509248
6,Q4,0,0.000297,27,91022
7,Q4,1,0.000595,303,508978
8,Q5_high,0,0.008956,778,86865
9,Q5_high,1,0.013380,6866,513135


In [13]:
import statsmodels.formula.api as smf

model = smf.ols(
    "conversion ~ treatment",
    data=df
).fit(cov_type="HC1")

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             conversion   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1123.
Date:                Sun, 09 Aug 2026   Prob (F-statistic):          3.27e-246
Time:                        18:53:56   Log-Likelihood:             2.0986e+07
No. Observations:            13979592   AIC:                        -4.197e+07
Df Residuals:                13979590   BIC:                        -4.197e+07
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0019   3.04e-05     63.804      0.0

In [14]:
overall = df.groupby("treatment")["conversion"].agg(["mean", "count", "sum"])
print(overall)

itt = (
    df.loc[df["treatment"] == 1, "conversion"].mean()
    - df.loc[df["treatment"] == 0, "conversion"].mean()
)

print("ITT:", itt)

               mean     count    sum
treatment                           
0          0.001938   2096937   4063
1          0.003089  11882655  36711
ITT: 0.0011518730521316279


In [15]:
model_visit = smf.ols(
    "visit ~ treatment",
    data=df
).fit(cov_type="HC1")

print(model_visit.summary())

                            OLS Regression Results                            
Dep. Variable:                  visit   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     4996.
Date:                Sun, 09 Aug 2026   Prob (F-statistic):               0.00
Time:                        18:56:20   Log-Likelihood:             1.8756e+06
No. Observations:            13979592   AIC:                        -3.751e+06
Df Residuals:                13979590   BIC:                        -3.751e+06
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0382      0.000    288.594      0.0